In [1]:
from glob import glob
import pandas as pd
import os
import soundfile as sf
from tqdm import tqdm
from multiprocess import Pool
import librosa
import itertools
import io
import numpy as np
import json

def chunks(l, n):
    for i in range(0, len(l), n):
        yield (l[i: i + n], i // n)

def multiprocessing(strings, function, cores=6, returned=True):
    df_split = chunks(strings, len(strings) // cores)
    pool = Pool(cores)
    pooled = pool.map(function, df_split)
    pool.close()
    pool.join()

    if returned:
        return list(itertools.chain(*pooled))

In [20]:
files = glob('ClArTTS/data/*')
f = files[0]
base = f.split('/')[0] + '_audio'
f_new = f.replace('/', '-').replace('.parquet', '')
base, f_new

('ClArTTS_audio', 'ClArTTS-data-train-00015-of-00026-5a1bba801b33852b')

In [28]:
def loop(files):

    os.environ['OMP_NUM_THREADS'] = '1'
    os.environ['OPENBLAS_NUM_THREADS'] = '1'

    files, _ = files

    data = []
    for f in tqdm(files):
        base = f.split('/')[0] + '_audio'
        f_new = f.replace('/', '-').replace('.parquet', '')
        os.makedirs(base, exist_ok=True)
        df = pd.read_parquet(f)
        for i in range(len(df)):
            try:
                t = df['text'].iloc[i].strip()
                if len(t) < 2:
                    continue
                audio_filename = f'{f_new}_{i}.mp3'
                audio_filename = os.path.join(base, audio_filename)
                audio_np = df['audio'].iloc[i]
                audio_np = librosa.resample(audio_np, orig_sr = df['sampling_rate'].iloc[i], 
                                            target_sr = 24000)
                sf.write(audio_filename, audio_np, 24000)
                
                data.append({
                    'audio_filename': audio_filename,
                    'text': df['text'].iloc[i],
                    'speaker': f"ClArTTS"
                })
            except Exception as e:
                print(e)
                pass
        
    return data

In [29]:
data = loop((files[:1], 0))

  0%|          | 0/1 [00:00<?, ?it/s]/usr/lib/python3/dist-packages/scipy/__init__.py:146: UserWarning: A NumPy version >=1.17.3 and <1.25.0 is required for this version of SciPy (detected version 1.26.4
  warnings.warn(f"A NumPy version >={np_minversion} and <{np_maxversion}"
100%|██████████| 1/1 [00:14<00:00, 14.95s/it]


In [30]:
len(data)

365

In [31]:
data[0]

{'audio_filename': 'ClArTTS_audio/ClArTTS-data-train-00015-of-00026-5a1bba801b33852b_0.mp3',
 'text': ':وَقَدْ حُكِيَ أَنَّ الْحَجَّاجَ قَالَ لِأَعْرَابِيٍّ أَخَطِيبٌ أَنَا؟',
 'speaker': 'ClArTTS'}

In [32]:
import IPython.display as ipd
ipd.Audio(data[0]['audio_filename'])

In [34]:
data = multiprocessing(files, loop, cores = len(files))

100%|██████████| 1/1 [00:15<00:00, 15.53s/it]


In [35]:
len(data)

9705

In [36]:
from datasets import Dataset

dataset = Dataset.from_list(data)
dataset[0]

/home/ubuntu/.local/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


{'audio_filename': 'ClArTTS_audio/ClArTTS-data-train-00015-of-00026-5a1bba801b33852b_0.mp3',
 'text': ':وَقَدْ حُكِيَ أَنَّ الْحَجَّاجَ قَالَ لِأَعْرَابِيٍّ أَخَطِيبٌ أَنَا؟',
 'speaker': 'ClArTTS'}

In [37]:
dataset.push_to_hub('malaysia-ai/Multilingual-TTS', 'ClArTTS')

Creating parquet from Arrow format: 100%|██████████| 1/1 [00:00<00:00, 170.81ba/s]
Processing Files (0 / 0): |          |  0.00B /  0.00B            
Processing Files (0 / 1):  77%|███████▋  |  566kB /  735kB, 2.83MB/s  
Processing Files (1 / 1): 100%|██████████|  735kB /  735kB, 1.84MB/s  
Processing Files (1 / 1): 100%|██████████|  735kB /  735kB, 1.84MB/s  
New Data Upload: 100%|██████████|  735kB /  735kB, 1.84MB/s  
Uploading the dataset shards: 100%|██████████| 1/1 [00:00<00:00,  1.23 shards/s]


CommitInfo(commit_url='https://huggingface.co/datasets/malaysia-ai/Multilingual-TTS/commit/a5c5115dd3757707d7fbcdd39be99325e2b995d6', commit_message='Upload dataset', commit_description='', oid='a5c5115dd3757707d7fbcdd39be99325e2b995d6', pr_url=None, repo_url=RepoUrl('https://huggingface.co/datasets/malaysia-ai/Multilingual-TTS', endpoint='https://huggingface.co', repo_type='dataset', repo_id='malaysia-ai/Multilingual-TTS'), pr_revision=None, pr_num=None)

In [38]:
audio_files = [d['audio_filename'] for d in data]

with open('ClArTTS-audio.json', 'w') as fopen:
    json.dump(list(set(audio_files)), fopen)

In [40]:
folders = glob('ClArTTS_audio*')
folders = [f for f in folders if '.zip' not in f]
for f in folders:
    print(f)
    os.system(f'zip -rq {f}.zip {f}')

ClArTTS_audio_neucodec
ClArTTS_audio


In [41]:
from huggingface_hub import HfApi
api = HfApi()

for f in glob('ClArTTS_audio*.zip'):
    api.upload_file(
        path_or_fileobj=f,
        path_in_repo=f,
        repo_id="malaysia-ai/Multilingual-TTS",
        repo_type="dataset",
    )

Processing Files (0 / 0): |          |  0.00B /  0.00B            
Processing Files (0 / 1):  13%|█▎        | 39.8MB /  313MB,   ???B/s  
Processing Files (0 / 1):  54%|█████▎    |  168MB /  313MB,  644MB/s  
Processing Files (0 / 1):  99%|█████████▉|  311MB /  313MB,  679MB/s  
Processing Files (0 / 1): 100%|█████████▉|  312MB /  313MB,  272MB/s  
Processing Files (0 / 1): 100%|█████████▉|  313MB /  313MB,  227MB/s  
Processing Files (1 / 1): 100%|██████████|  313MB /  313MB,  171MB/s  
Processing Files (1 / 1): 100%|██████████|  313MB /  313MB,  137MB/s  
New Data Upload: 100%|██████████|  313MB /  313MB,  137MB/s  
Processing Files (0 / 0): |          |  0.00B /  0.00B            
Processing Files (0 / 1):  95%|█████████▌| 8.70MB / 9.12MB,   ???B/s  
Processing Files (1 / 1): 100%|██████████| 9.12MB / 9.12MB, 2.08MB/s  
Processing Files (1 / 1): 100%|██████████| 9.12MB / 9.12MB, 1.04MB/s  
New Data Upload: 100%|██████████| 9.12MB / 9.12MB, 1.04MB/s  
